# Module 11 Lab: End-to-End Agentic RAG for IT Infrastructure Operations

This notebook builds an **IT Infrastructure Operations AI Assistant**. Run the cells in order, from top to bottom. Each section has a description, the code, an explanation and the expected output.

**You will build an agent that answers operations questions by combining:**
- a **vector knowledge base** (IT procedures searched by meaning, using FAISS), and
- **Excel custom tools** (server, incident, service and summary data, read with Pandas).

**Before you start:** complete the setup steps in the README (Python packages and a `.env` file with your OpenAI key). Start Jaeger (Section 10) only when you want to see traces.

## Section 1: Introduction to Agentic RAG

**Description.** Read this before running any code.

**Traditional RAG** (Retrieval-Augmented Generation) searches a document library for text related to a question, adds that text to the prompt, and asks a language model (LLM) to write the answer. The path is always the same: search once, then answer.

**Agentic RAG** puts the LLM in charge of the steps. The agent decides which sources to use, calls them as tools, reads the results and decides whether another step is needed. One question can use a document search and a spreadsheet lookup before the answer is written.

**Custom tools** are ordinary Python functions that the agent is allowed to call. Each tool has a name and a plain-language description. The LLM reads those descriptions and chooses when to call each tool. The tools in this lab only read data.

**Structured and unstructured data** need different tools:
- *Unstructured* data (procedures written as text) is searched by meaning with vector search.
- *Structured* data (rows and columns in Excel) is filtered with Pandas.

**Architecture of this lab:**

```
User question
     |
     v
Agent: the LLM chooses the tools and their order
     |-- search_knowledge_base       vector search over IT procedures (FAISS)
     |-- search_server_details       Excel: Infrastructure sheet
     |-- search_incidents            Excel: Incidents sheet
     |-- get_service_details         Excel: Service_Inventory sheet
     |-- get_infrastructure_summary  Excel: counts and utilization summary
     |
     v
Final answer: facts, guidance and recommendations, labelled separately
     |
     v
Evaluation (quality scores)  +  OpenTelemetry traces (viewed in Jaeger)
```

## Section 2: Environment Setup

**Description.** Check that the required packages are installed, load your OpenAI key from the `.env` file, and create the chat model and the embedding model. This section does not call OpenAI yet.

- `load_dotenv` reads the `.env` file and makes its values available to the program. The key is never printed.
- `ChatOpenAI` is the LLM. `temperature=0` makes answers more repeatable.
- `OpenAIEmbeddings` turns text into vectors (lists of numbers). Texts with similar meaning get similar vectors.
- If the key is missing, the cell stops with a clear message instead of failing later in the lab.

In [ ]:
import os
import json
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import pandas as pd
from dotenv import load_dotenv

# Packages from requirements.txt. Each one must be installed in this interpreter.
PACKAGES = ['langchain', 'langchain-core', 'langchain-openai', 'langchain-community',
            'openai', 'faiss-cpu', 'pandas', 'openpyxl', 'opentelemetry-sdk']

for name in PACKAGES:
    try:
        print(name.ljust(24), 'version', version(name))
    except PackageNotFoundError:
        print(name.ljust(24), 'NOT INSTALLED. Run: python -m pip install -r requirements.txt')

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

ENV_FILE = Path.cwd() / '.env'
load_dotenv(ENV_FILE)  # copies the values in .env into environment variables

OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
EMBEDDING_MODEL = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')
SERVICE_NAME = os.getenv('OTEL_SERVICE_NAME', 'it-infrastructure-agent')
OTLP_ENDPOINT = os.getenv('OTEL_EXPORTER_OTLP_ENDPOINT', 'http://localhost:4318')

api_key = os.getenv('OPENAI_API_KEY', '')
if api_key in ('', 'your_openai_api_key_here'):
    raise RuntimeError(
        'OPENAI_API_KEY is missing or still the placeholder. '
        'Copy .env.example to .env, add your key, restart the kernel, and run this cell again.'
    )

llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)   # temperature 0 = most repeatable answers
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)   # turns text into vectors (lists of numbers)

print('OpenAI key found (value hidden).')
print('Chat model:      ', OPENAI_MODEL)
print('Embedding model: ', EMBEDDING_MODEL)
print('Trace endpoint:  ', OTLP_ENDPOINT)

## Section 3: Prepare the Excel Dataset

**Description.** The workbook `it_infrastructure_data.xlsx` holds all of the structured data. All records are fictional. The next cells create the workbook if it is missing, load its three worksheets and check the columns.

| Worksheet | One row is | Key columns and allowed values |
|---|---|---|
| Infrastructure | one server | Server_ID, Server_Name, Environment (Production, UAT, Development), CPU / Memory / Disk_Utilization (percent), Server_Status (Running, Warning, Critical), Application, Owner_Team, Location |
| Incidents | one incident | Incident_ID, Server_ID, Incident_Title, Category, Severity (Low, Medium, High, Critical), Status (Open, In Progress, Resolved), Assigned_Team, Created_Date, Resolution_Notes |
| Service_Inventory | one business service | Service_ID, Service_Name, Server_ID, Business_Criticality (High, Medium, Low), Service_Status (Available, Degraded, Unavailable), Support_Team |

**Status rule used for the sample data:** a server is **Critical** if any of its utilization values is 95 percent or more, **Warning** if any is 85 percent or more, and otherwise **Running**.

In [ ]:
# The sample records. Each inner list is one row, in the same order as the column list.
INFRA_COLUMNS = ['Server_ID', 'Server_Name', 'Environment', 'Operating_System',
                 'CPU_Utilization', 'Memory_Utilization', 'Disk_Utilization',
                 'Server_Status', 'Application', 'Owner_Team', 'Location']

INFRA_ROWS = [
    ['SRV-101', 'APP-WEB-01', 'Production', 'Windows Server 2022', 42, 55, 61, 'Running', 'Customer Portal', 'Web Platform Team', 'DC-North'],
    ['SRV-102', 'APP-ORDERS-02', 'Production', 'Windows Server 2019', 58, 93, 70, 'Warning', 'Order Management', 'Application Team', 'DC-North'],
    ['SRV-103', 'DB-FIN-01', 'Production', 'Red Hat Enterprise Linux 9', 35, 71, 88, 'Warning', 'Finance Database', 'Database Team', 'DC-North'],
    ['SRV-104', 'WEB-DMZ-01', 'Production', 'Ubuntu 22.04 LTS', 22, 40, 38, 'Running', 'Public Website', 'Network and Web Team', 'Cloud-Region-A'],
    ['SRV-105', 'APP-PAY-01', 'Production', 'Red Hat Enterprise Linux 8', 96, 89, 79, 'Critical', 'Payment Gateway', 'Payments Platform Team', 'DC-South'],
    ['SRV-106', 'FILE-HR-01', 'UAT', 'Windows Server 2019', 18, 49, 92, 'Warning', 'HR File Share', 'End User Computing Team', 'DC-South'],
    ['SRV-107', 'APP-CRM-UAT-01', 'UAT', 'Windows Server 2022', 30, 62, 45, 'Running', 'CRM Test Environment', 'Application Team', 'Cloud-Region-B'],
    ['SRV-108', 'DB-INV-DEV-01', 'Development', 'Windows Server 2022', 64, 88, 55, 'Warning', 'Inventory Dev Database', 'Database Team', 'Cloud-Region-B'],
    ['SRV-109', 'MON-LOG-01', 'Production', 'Ubuntu 22.04 LTS', 27, 66, 97, 'Critical', 'Central Logging', 'Monitoring Team', 'DC-North'],
    ['SRV-110', 'AUTH-SSO-01', 'Production', 'Red Hat Enterprise Linux 9', 15, 35, 42, 'Running', 'Single Sign-On', 'Identity Team', 'DC-South'],
    ['SRV-111', 'BKP-VAULT-01', 'Production', 'Windows Server 2019', 12, 30, 86, 'Warning', 'Backup Services', 'Backup Team', 'DC-North'],
    ['SRV-112', 'APP-SCM-DEV-02', 'Development', 'Ubuntu 20.04 LTS', 8, 28, 33, 'Running', 'Supply Chain Dev', 'Application Team', 'Cloud-Region-B'],
    ['SRV-113', 'DB-RPT-UAT-01', 'UAT', 'Ubuntu 22.04 LTS', 52, 58, 67, 'Running', 'Reporting Database', 'Database Team', 'Cloud-Region-A'],
    ['SRV-114', 'APP-API-01', 'Production', 'Red Hat Enterprise Linux 9', 97, 76, 58, 'Critical', 'Partner API Gateway', 'Integration Team', 'Cloud-Region-A'],
]

INCIDENT_COLUMNS = ['Incident_ID', 'Server_ID', 'Incident_Title', 'Category', 'Severity',
                    'Status', 'Assigned_Team', 'Created_Date', 'Resolution_Notes']

INCIDENT_ROWS = [
    ['INC-2001', 'SRV-102', 'Order service memory above 90 percent', 'Application', 'High', 'In Progress', 'Application Team', '2026-11-18', 'Order service pool restarted once. Memory climbs again within hours. Heap analysis requested.'],
    ['INC-2002', 'SRV-102', 'Slow order search responses', 'Application', 'Medium', 'Resolved', 'Application Team', '2026-10-30', 'Stale cache entries cleared. Response times back to normal.'],
    ['INC-2003', 'SRV-105', 'Payment authorization failures', 'Application', 'Critical', 'Open', 'Payments Platform Team', '2026-11-20', 'Under investigation. Escalated to the on-call duty manager.'],
    ['INC-2004', 'SRV-105', 'CPU saturation during batch settlement', 'Application', 'Critical', 'In Progress', 'Payments Platform Team', '2026-11-21', 'Settlement batch moved to off-peak hours. CPU still above 90 percent.'],
    ['INC-2005', 'SRV-105', 'Intermittent TLS handshake errors', 'Network', 'High', 'Resolved', 'Network Team', '2026-11-02', 'Load balancer cipher policy updated. No errors since.'],
    ['INC-2006', 'SRV-103', 'Finance database disk space warning', 'Storage', 'High', 'Open', 'Database Team', '2026-11-19', 'Archive job waiting for change approval.'],
    ['INC-2007', 'SRV-109', 'Central logging disk almost full', 'Storage', 'Critical', 'In Progress', 'Monitoring Team', '2026-11-22', 'Old log files being compressed and moved to archive storage.'],
    ['INC-2008', 'SRV-106', 'HR file share almost full', 'Storage', 'Medium', 'Open', 'End User Computing Team', '2026-11-15', 'Waiting for users to remove old files.'],
    ['INC-2009', 'SRV-111', 'Nightly backup job failed', 'Backup', 'High', 'Resolved', 'Backup Team', '2026-11-10', 'Backup agent restarted and job re-run successfully.'],
    ['INC-2010', 'SRV-114', 'Partner API timeouts', 'Application', 'Critical', 'Open', 'Integration Team', '2026-11-23', 'Traffic spike under investigation. CPU at 97 percent.'],
    ['INC-2011', 'SRV-108', 'Development database connection refused', 'Database', 'Low', 'Resolved', 'Database Team', '2026-11-05', 'Connection pool limit increased in the development environment.'],
    ['INC-2012', 'SRV-113', 'Reporting query timeouts', 'Database', 'Medium', 'In Progress', 'Database Team', '2026-11-17', 'Query plan review in progress.'],
    ['INC-2013', 'SRV-101', 'Customer portal login page slow', 'Network', 'Low', 'Resolved', 'Web Platform Team', '2026-10-28', 'DNS cache flushed. Latency back to normal.'],
    ['INC-2014', 'SRV-110', 'Single sign-on certificate expiry warning', 'Security', 'Medium', 'Open', 'Identity Team', '2026-11-21', 'Certificate renewal scheduled for the next change window.'],
]

SERVICE_COLUMNS = ['Service_ID', 'Service_Name', 'Server_ID', 'Business_Criticality',
                   'Service_Status', 'Support_Team']

SERVICE_ROWS = [
    ['SVC-501', 'Order Management API', 'SRV-102', 'High', 'Degraded', 'Application Team'],
    ['SVC-502', 'Payment Authorization', 'SRV-105', 'High', 'Degraded', 'Payments Platform Team'],
    ['SVC-503', 'Finance Reporting Database', 'SRV-103', 'High', 'Available', 'Database Team'],
    ['SVC-504', 'Customer Portal', 'SRV-101', 'High', 'Available', 'Web Platform Team'],
    ['SVC-505', 'Central Log Search', 'SRV-109', 'Medium', 'Degraded', 'Monitoring Team'],
    ['SVC-506', 'Single Sign-On', 'SRV-110', 'High', 'Available', 'Identity Team'],
    ['SVC-507', 'Partner API Gateway', 'SRV-114', 'High', 'Degraded', 'Integration Team'],
    ['SVC-508', 'Nightly Backup Service', 'SRV-111', 'Medium', 'Available', 'Backup Team'],
    ['SVC-509', 'Developer Inventory Database', 'SRV-108', 'Low', 'Unavailable', 'Database Team'],
]

In [ ]:
DATA_FILE = Path.cwd() / 'it_infrastructure_data.xlsx'

# Create the workbook only if it is missing. Each DataFrame becomes one worksheet.
if not DATA_FILE.exists():
    print('Workbook not found. Creating it from the sample data...')
    with pd.ExcelWriter(DATA_FILE, engine='openpyxl') as writer:
        pd.DataFrame(INFRA_ROWS, columns=INFRA_COLUMNS).to_excel(writer, sheet_name='Infrastructure', index=False)
        pd.DataFrame(INCIDENT_ROWS, columns=INCIDENT_COLUMNS).to_excel(writer, sheet_name='Incidents', index=False)
        pd.DataFrame(SERVICE_ROWS, columns=SERVICE_COLUMNS).to_excel(writer, sheet_name='Service_Inventory', index=False)
    print('Created', DATA_FILE.name)
else:
    print('Workbook found:', DATA_FILE.name)

# Load all three worksheets. sheet_name=None returns a dictionary of DataFrames.
sheets = pd.read_excel(DATA_FILE, sheet_name=None, engine='openpyxl')

INFRA_DF = sheets['Infrastructure']
INCIDENTS_DF = sheets['Incidents']
SERVICES_DF = sheets['Service_Inventory']

# Validation: every required column must exist in its worksheet.
REQUIRED = {
    'Infrastructure': INFRA_COLUMNS,
    'Incidents': INCIDENT_COLUMNS,
    'Service_Inventory': SERVICE_COLUMNS,
}
for sheet_name, columns in REQUIRED.items():
    missing = [c for c in columns if c not in sheets[sheet_name].columns]
    if missing:
        raise ValueError(f'{sheet_name} is missing columns: {missing}. Delete the workbook and run this cell again.')
print('All required columns were found in all three worksheets.')

In [ ]:
print('Worksheets found:', list(sheets.keys()))
print('Rows: Infrastructure =', len(INFRA_DF), '| Incidents =', len(INCIDENTS_DF), '| Service_Inventory =', len(SERVICES_DF))

print('\nInfrastructure (first 3 rows):')
display(INFRA_DF.head(3))
print('\nIncidents (first 3 rows):')
display(INCIDENTS_DF.head(3))
print('\nService_Inventory (first 3 rows):')
display(SERVICES_DF.head(3))

print('\nServers by status:')
print(INFRA_DF['Server_Status'].value_counts().to_string())
print('\nServers by environment:')
print(INFRA_DF['Environment'].value_counts().to_string())

open_incidents = int((INCIDENTS_DF['Status'] != 'Resolved').sum())
print('\nIncidents not yet resolved:', open_incidents)

## Section 4: Build the Vector Knowledge Base

**Description.** Vector search finds documents by meaning. A question can match a procedure even when the words are different.

- **Embedding:** a vector (a list of numbers) that represents the meaning of a text. Similar meanings give vectors that point in similar directions.
- **Vector search:** the question is turned into a vector and compared with every document vector. The closest documents are returned.
- **FAISS:** a fast index that holds the vectors in memory. It needs no database server.
- **Document:** a LangChain object with `page_content` (the text) and `metadata` (the title and an id).
- **Distance:** FAISS returns a distance. A **lower** distance means a **closer** match.

The knowledge base is 12 short documents. Each one has an issue, symptoms, diagnostic steps, a recommended action and escalation guidance.

In [ ]:
KB_DOCS = [
    {
        'title': 'High CPU Utilization Troubleshooting',
        'issue': 'A server runs at high CPU utilization, usually above 90 percent for 15 minutes or more.',
        'symptoms': 'Slow responses, request timeouts, queued work and a high CPU reading on the monitoring dashboard.',
        'diagnostics': 'Identify the top CPU processes. Check whether a scheduled job or batch run started at the same time. Check for a recent deployment.',
        'action': 'Reschedule non-urgent jobs. Stop a runaway process only after the owner agrees. Add capacity if the high load is expected to continue.',
        'escalation': 'If CPU stays above 90 percent for 30 minutes, or customers are affected, escalate to the owning application team and the on-call manager.',
    },
    {
        'title': 'High Memory Utilization Troubleshooting',
        'issue': 'Memory utilization is high, for example above 90 percent, and may keep rising.',
        'symptoms': 'Slow application response, heavy paging to disk, and out-of-memory warnings in the event or application log.',
        'diagnostics': 'Find the processes with the largest memory use. Check whether memory keeps growing after a restart, which suggests a leak. Review recent changes and the application log for memory errors.',
        'action': 'Capture a memory dump or heap report before any restart so the owner can analyze it. Restart the affected service in a planned window if business impact allows. Clear caches that can be rebuilt safely. Confirm that memory returns to a normal level.',
        'escalation': 'If memory is above 90 percent again within a few hours of a restart, or the service is degraded, escalate to the application team to investigate a possible leak. Inform the on-call manager for Critical services.',
    },
    {
        'title': 'Disk Space Management',
        'issue': 'A volume is almost full, for example above 85 percent, which can stop logs, databases and backups.',
        'symptoms': 'Write errors, failed backups, low free space warnings, and services that stop saving data.',
        'diagnostics': 'Find the largest folders and files. Check log growth and old backup copies. Check whether a database transaction log is full.',
        'action': 'Remove or compress logs that are past their retention period, following the retention policy. Move old archives to storage. Never delete data that has no approved retention rule. Record what was removed.',
        'escalation': 'If free space falls below 5 percent, or a database or backup is affected, escalate to the storage or database team. Raise a change request for any permanent capacity increase.',
    },
    {
        'title': 'Server Availability and Restart Procedure',
        'issue': 'A server does not respond to ping, remote login or health checks.',
        'symptoms': 'A monitoring alert for server down, no remote session, and service endpoints that do not respond.',
        'diagnostics': 'Check the hypervisor or cloud console for the server state. Check network reachability from a second location. Check the last change or patch window.',
        'action': 'If the server is hung and a restart is approved, use the standard restart procedure and inform the application owner first. Confirm that services start and health checks pass. Review the system log after the restart to find the cause.',
        'escalation': 'If the server does not return within 15 minutes, or it is a Critical server, escalate to the infrastructure operations team and the owning team.',
    },
    {
        'title': 'Network Connectivity Troubleshooting',
        'issue': 'A server cannot reach another server or service, or users cannot reach an application.',
        'symptoms': 'Timeouts, connection refused messages, or intermittent slowness depending on the route.',
        'diagnostics': 'Check that the server has an address and a default gateway. Test name resolution. Test the port the application needs. Check the firewall rule and the load balancer status.',
        'action': 'Restore the correct route or firewall rule through the approved change process. Restart a network service only if it is approved. Verify the connection from both ends.',
        'escalation': 'If the fault affects more than one server or a whole site, escalate to the network team at once and inform the incident manager.',
    },
    {
        'title': 'Database Connectivity Troubleshooting',
        'issue': 'An application cannot connect to its database, or connections fail intermittently.',
        'symptoms': 'Connection refused, login failed, timeouts, or a message that the connection pool is exhausted.',
        'diagnostics': 'Check that the database service is running. Compare the connection count with the limit. Check the database listener and port. Check whether credentials were recently changed or have expired.',
        'action': 'Restart the application connection pool if approved. Ask the database team to raise the pool limit or clear idle sessions. Do not share or change credentials in chat. Use the approved secrets process.',
        'escalation': 'If the database is unavailable or a Critical service is affected, escalate to the database team and the on-call manager.',
    },
    {
        'title': 'Application Service Failure',
        'issue': 'An application service has stopped or keeps crashing.',
        'symptoms': 'Service status shows stopped or restarting, errors in the application log, and users see error pages.',
        'diagnostics': 'Check the service status and the latest application log entries. Check dependencies such as the database and the network. Check recent deployments and configuration changes.',
        'action': 'Restart the service once if the failure looks transient and the owner agrees. Roll back the latest change if it is the likely cause. Record the exact error text for the owner.',
        'escalation': 'If the service fails again after one restart, or it is business critical, escalate to the application owner team and log an incident with severity High or Critical.',
    },
    {
        'title': 'Incident Severity Classification',
        'issue': 'Each incident needs a severity so that the right team responds at the right speed.',
        'symptoms': 'Severity is based on business impact, not on a single technical symptom.',
        'diagnostics': 'Ask: Is a business-critical service down or degraded? How many users are affected? Is there a security or data risk? Is there a workaround?',
        'action': 'Critical: a business-critical service is down, or there is a security or data-loss risk. High: a service is degraded with no workaround. Medium: a degraded service with a workaround. Low: a minor issue with little business impact.',
        'escalation': 'Change the severity when the impact changes. Critical and High incidents need a named owner within 15 minutes.',
    },
    {
        'title': 'Incident Escalation Procedure',
        'issue': 'An incident needs help beyond the first responder.',
        'symptoms': 'The fix is not known, the first responder is blocked, or the impact is growing.',
        'diagnostics': 'Confirm the severity. Note what has been tried and when. Identify the owning team from the service inventory.',
        'action': 'Escalate in this order: the owning team lead, then the on-call manager for Critical incidents. Send a short update with the server, symptom, impact, steps tried and next step. Keep the incident record current.',
        'escalation': 'Critical incidents go to the on-call manager at once. Escalate again if no update is given within 30 minutes.',
    },
    {
        'title': 'Backup Failure Troubleshooting',
        'issue': 'A scheduled backup job failed or did not finish.',
        'symptoms': 'Failed job status in the backup console, missing recovery points, or backup agent errors.',
        'diagnostics': 'Read the job error message. Check that the backup agent service is running. Check free space on the backup target. Check that the source server is reachable.',
        'action': 'Restart the backup agent if approved. Free space on the backup target through the approved clean-up process. Run the job again and confirm that it completes. Check that the newest restore point can be used.',
        'escalation': 'If two runs fail in a row, or business-critical data has no recent recovery point, escalate to the backup team and the owning application or database team.',
    },
    {
        'title': 'Monitoring and Alert Response',
        'issue': 'A monitoring alert fires and someone must decide what to do.',
        'symptoms': 'Alerts for CPU, memory, disk, service status or server availability.',
        'diagnostics': 'Check the alert time and the current metric value. Check whether the alert is a known issue or planned work. Check for duplicate alerts.',
        'action': 'Acknowledge the alert. Follow the matching troubleshooting document. Record the outcome in the incident or alert log. Change alert thresholds only through the monitoring team.',
        'escalation': 'If the alert is for a Critical server and the cause is still unknown after 15 minutes, raise an incident and escalate.',
    },
    {
        'title': 'Basic Server Health Checks',
        'issue': 'A regular check confirms that a server is healthy.',
        'symptoms': 'Not applicable. This is a routine check.',
        'diagnostics': 'Check CPU, memory and disk utilization. Check that critical services are running. Check the last successful backup. Check for pending security updates.',
        'action': 'Record the results. Treat utilization of 85 percent or more as Warning, and 95 percent or more as Critical, which is the status rule used for the lab data. Raise an incident for any Critical result.',
        'escalation': 'Report repeated Warning results for the same server to the owning team at the weekly review.',
    },
]

print('Knowledge-base documents defined:', len(KB_DOCS))

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

# Convert each knowledge-base entry into a LangChain Document.
docs = []
for number, item in enumerate(KB_DOCS, start=1):
    text = '\n'.join([
        item['title'],
        'Issue: ' + item['issue'],
        'Symptoms: ' + item['symptoms'],
        'Diagnostic steps: ' + item['diagnostics'],
        'Recommended action: ' + item['action'],
        'Escalation: ' + item['escalation'],
    ])
    docs.append(Document(page_content=text, metadata={'doc_id': number, 'title': item['title']}))

print('Documents prepared:', len(docs))
print('Creating embeddings for all documents and building the FAISS index (calls the OpenAI embedding API)...')

# from_documents sends each text to the embedding model and stores the vectors in FAISS.
vector_store = FAISS.from_documents(docs, embeddings)
print('Vectors stored in the FAISS index:', vector_store.index.ntotal)

In [ ]:
def retrieve_knowledge(query, k=3):
    '''Return the k closest knowledge-base documents for a question.'''
    pairs = vector_store.similarity_search_with_score(query, k=k)
    results = []
    for doc, distance in pairs:
        results.append({
            'title': doc.metadata['title'],
            'distance': round(float(distance), 4),
            'text': doc.page_content,
        })
    return results

test_query = 'Server memory is almost full and the application is slow'
hits = retrieve_knowledge(test_query, k=3)

print('Test query:', test_query)
hit_table = pd.DataFrame([
    {'rank': rank, 'title': hit['title'], 'distance (lower = closer)': hit['distance']}
    for rank, hit in enumerate(hits, start=1)
])
display(hit_table)

**Expected output:** `Vectors stored in the FAISS index: 12`. The table lists three documents with their distances. Memory-related documents usually rank first for this query, and the distances are small decimal numbers. Exact values can vary slightly between runs.

## Section 5: Build the Excel Custom Tools

**Description.** Four read-only tools read the Excel workbook. Each tool is a plain Python function with a short, clear docstring.

**How a Python function becomes an agent tool:**
1. The `@tool` line above a function tells LangChain to register it as a tool.
2. LangChain reads the function name, the type hints (for example `server: str`) and the docstring. These become the tool's name, its input fields and its description.
3. The LLM receives that description. It decides whether to call the tool, and with which input, from the user's question.
4. When the LLM asks for a call, LangChain runs the function and returns the text result to the LLM.

**Safety rules for these tools:** they only read data, with Pandas filtering. They never change the workbook. They never run code or SQL that the LLM writes. Inputs are trimmed, length-limited and checked against the allowed values. Missing records return a clear message.

In [ ]:
from opentelemetry import trace
from langchain_core.tools import tool

# A tracer creates OpenTelemetry spans. Until Section 9 configures an exporter,
# the spans are created but not sent anywhere. That is expected at this point.
tracer = trace.get_tracer(SERVICE_NAME)

MAX_TEXT_LENGTH = 60
SEVERITIES = ['Low', 'Medium', 'High', 'Critical']
INCIDENT_STATUSES = ['Open', 'In Progress', 'Resolved']
ENVIRONMENTS = ['Production', 'UAT', 'Development']
SERVER_STATUSES = ['Running', 'Warning', 'Critical']


def clean_text(value, limit=MAX_TEXT_LENGTH):
    '''Turn tool input into trimmed text with a length limit.'''
    text = str(value or '').strip()
    return text[:limit]


def format_record(row):
    '''Show one record as Field: value lines.'''
    return '\n'.join('  ' + str(col) + ': ' + str(row[col]) for col in row.index)


def format_records(frame):
    '''Format every row of a DataFrame, with a blank line between records.'''
    return '\n\n'.join(format_record(row) for _, row in frame.iterrows())


@tool
def search_server_details(server: str) -> str:
    '''Look up one server in the Infrastructure worksheet by Server_ID (for example SRV-102) or by Server_Name. Read-only.'''
    with tracer.start_as_current_span('Excel Server Lookup') as span:
        span.set_attribute('tool.name', 'search_server_details')
        key = clean_text(server).upper()
        if not key:
            span.set_attribute('tool.matches', 0)
            return 'Please give a Server_ID such as SRV-102, or a Server_Name.'

        ids = INFRA_DF['Server_ID'].astype(str).str.upper()
        names = INFRA_DF['Server_Name'].astype(str).str.upper()
        matches = INFRA_DF[(ids == key) | (names == key)]
        span.set_attribute('tool.matches', len(matches))

        if matches.empty:
            return 'No server found for ' + key + '. Check the Server_ID or Server_Name.'
        return 'Found ' + str(len(matches)) + ' server(s):\n\n' + format_records(matches)

In [ ]:
@tool
def search_incidents(server_id: str = '', incident_id: str = '', severity: str = '', status: str = '') -> str:
    '''Search the Incidents worksheet. Combine any of: server_id (for example SRV-105), incident_id (for example INC-2003), severity (Low, Medium, High, Critical) and status (Open, In Progress, Resolved). Read-only.'''
    with tracer.start_as_current_span('Excel Incident Lookup') as span:
        span.set_attribute('tool.name', 'search_incidents')
        server_key = clean_text(server_id).upper()
        incident_key = clean_text(incident_id).upper()
        severity_key = clean_text(severity).lower()
        status_key = clean_text(status).lower()

        if not (server_key or incident_key or severity_key or status_key):
            span.set_attribute('tool.matches', 0)
            return 'Give at least one filter: server_id, incident_id, severity or status.'

        if severity_key and severity_key not in [s.lower() for s in SEVERITIES]:
            return 'Unknown severity ' + severity_key + '. Use one of: ' + ', '.join(SEVERITIES) + '.'
        if status_key and status_key not in [s.lower() for s in INCIDENT_STATUSES]:
            return 'Unknown status ' + status_key + '. Use one of: ' + ', '.join(INCIDENT_STATUSES) + '.'

        frame = INCIDENTS_DF
        if server_key:
            frame = frame[frame['Server_ID'].astype(str).str.upper() == server_key]
        if incident_key:
            frame = frame[frame['Incident_ID'].astype(str).str.upper() == incident_key]
        if severity_key:
            frame = frame[frame['Severity'].astype(str).str.lower() == severity_key]
        if status_key:
            frame = frame[frame['Status'].astype(str).str.lower() == status_key]
        span.set_attribute('tool.matches', len(frame))

        if frame.empty:
            return 'No incidents match those filters.'
        return 'Found ' + str(len(frame)) + ' incident(s):\n\n' + format_records(frame)

In [ ]:
@tool
def get_service_details(service: str) -> str:
    '''Look up business services in the Service_Inventory worksheet by Service_ID (for example SVC-502) or by part of the Service_Name (for example Payment). Read-only.'''
    with tracer.start_as_current_span('Excel Service Lookup') as span:
        span.set_attribute('tool.name', 'get_service_details')
        key = clean_text(service)
        if not key:
            span.set_attribute('tool.matches', 0)
            return 'Please give a Service_ID such as SVC-502, or part of a Service_Name.'

        by_id = SERVICES_DF['Service_ID'].astype(str).str.upper() == key.upper()
        by_name = SERVICES_DF['Service_Name'].astype(str).str.contains(key, case=False, regex=False)
        matches = SERVICES_DF[by_id | by_name]
        span.set_attribute('tool.matches', len(matches))

        if matches.empty:
            return 'No service found for ' + key + '.'
        return 'Found ' + str(len(matches)) + ' service(s):\n\n' + format_records(matches)


@tool
def get_infrastructure_summary(environment: str = '', server_status: str = '') -> str:
    '''Summarize the Infrastructure worksheet: server counts by status, average CPU, memory and disk utilization, and the Critical and Warning servers. Optional filters: environment (Production, UAT, Development) and server_status (Running, Warning, Critical). Read-only.'''
    with tracer.start_as_current_span('Excel Infrastructure Summary') as span:
        span.set_attribute('tool.name', 'get_infrastructure_summary')
        env_key = clean_text(environment).lower()
        status_key = clean_text(server_status).lower()

        if env_key and env_key not in [e.lower() for e in ENVIRONMENTS]:
            return 'Unknown environment ' + env_key + '. Use one of: ' + ', '.join(ENVIRONMENTS) + '.'
        if status_key and status_key not in [s.lower() for s in SERVER_STATUSES]:
            return 'Unknown server status ' + status_key + '. Use one of: ' + ', '.join(SERVER_STATUSES) + '.'

        frame = INFRA_DF
        if env_key:
            frame = frame[frame['Environment'].astype(str).str.lower() == env_key]
        if status_key:
            frame = frame[frame['Server_Status'].astype(str).str.lower() == status_key]
        span.set_attribute('tool.matches', len(frame))

        if frame.empty:
            return 'No servers match those filters.'

        lines = ['Servers in scope: ' + str(len(frame)), 'Count by status:']
        for status_name, count in frame['Server_Status'].value_counts().items():
            lines.append('  ' + str(status_name) + ': ' + str(count))
        lines.append('Average utilization (percent):')
        for column in ['CPU_Utilization', 'Memory_Utilization', 'Disk_Utilization']:
            lines.append('  ' + column + ': ' + str(round(frame[column].mean(), 1)))
        critical = frame[frame['Server_Status'] == 'Critical']['Server_ID'].tolist()
        warning = frame[frame['Server_Status'] == 'Warning']['Server_ID'].tolist()
        lines.append('Critical servers: ' + (', '.join(critical) or 'none'))
        lines.append('Warning servers: ' + (', '.join(warning) or 'none'))
        return '\n'.join(lines)

In [ ]:
print('Tool names the LLM will see:')
for tool_item in [search_server_details, search_incidents, get_service_details, get_infrastructure_summary]:
    print(' -', tool_item.name)

print('\n--- search_server_details: SRV-102 ---')
print(search_server_details.invoke({'server': 'SRV-102'}))

print('\n--- search_server_details: SRV-999 (does not exist) ---')
print(search_server_details.invoke({'server': 'SRV-999'}))

print('\n--- search_incidents: Critical incidents for SRV-105 ---')
print(search_incidents.invoke({'server_id': 'SRV-105', 'severity': 'Critical'}))

print('\n--- get_service_details: Payment ---')
print(get_service_details.invoke({'service': 'Payment'}))

print('\n--- get_infrastructure_summary: Production only ---')
print(get_infrastructure_summary.invoke({'environment': 'Production'}))

**Expected output (summary):**
- `search_server_details` for SRV-102 returns one Production server on Windows Server 2019 with memory at 93 percent and status Warning.
- `search_server_details` for SRV-999 returns `No server found for SRV-999. Check the Server_ID or Server_Name.` The tool reports the missing record and does not invent one.
- `search_incidents` for SRV-105 with severity Critical returns INC-2003 and INC-2004.
- `get_service_details` for Payment returns SVC-502, Payment Authorization.
- `get_infrastructure_summary` for Production covers 9 servers: 3 Running, 3 Warning and 3 Critical. The Critical servers are SRV-105, SRV-109 and SRV-114.

## Section 6: Build the RAG Retrieval Tool

**Description.** Wrap the vector search from Section 4 as a tool named `search_knowledge_base`. The agent can then call it the same way it calls the Excel tools. Use it for procedure, troubleshooting and guidance questions. The tool's description tells the LLM when to choose it.

In [ ]:
@tool
def search_knowledge_base(question: str) -> str:
    '''Search the IT operations knowledge base by meaning (vector search). Use it for troubleshooting steps, procedures, escalation rules and guidance. Returns the closest documents with their distances.'''
    with tracer.start_as_current_span('RAG Knowledge Retrieval') as span:
        span.set_attribute('tool.name', 'search_knowledge_base')
        query = clean_text(question, limit=300)
        if not query:
            span.set_attribute('rag.documents_retrieved', 0)
            return 'Please give a question or topic to search for.'

        hits = retrieve_knowledge(query, k=3)
        span.set_attribute('rag.k', 3)
        span.set_attribute('rag.documents_retrieved', len(hits))
        if hits:
            span.set_attribute('rag.top_match', hits[0]['title'])

        lines = ['Top knowledge-base matches (lower distance = closer match):']
        for rank, hit in enumerate(hits, start=1):
            lines.append('')
            lines.append(str(rank) + '. ' + hit['title'] + ' (distance ' + str(hit['distance']) + ')')
            lines.append(hit['text'])
        return '\n'.join(lines)

In [ ]:
print(search_knowledge_base.name)
print(search_knowledge_base.invoke({'question': 'What should I check when a server has high memory usage?'}))

**Expected output:** a numbered list of three knowledge-base matches, each with its distance and full text. The memory document usually ranks first. This is the exact text the agent receives as the tool result.

## Section 7: Build the LangChain Agent

**Description.** The agent is the LLM plus the five tools. It works in a loop, and every step is visible in this notebook:
1. The LLM receives the system prompt, the user question and the tool descriptions.
2. The LLM either writes a final answer, or asks for one or more tool calls (tool name and input values).
3. The notebook runs each requested tool and sends the result back to the LLM.
4. The loop repeats until the LLM gives a final answer, or the step limit is reached.

**How tool selection works.** The LLM does not run code by itself. It reads the question, the instructions and each tool description, then returns a structured request such as `search_server_details(server='SRV-102')`. Clear tool descriptions and clear instructions are what make the choice correct.

**Why a written-out loop?** LangChain's `bind_tools` attaches the tools to the LLM. Writing the loop out means each LLM turn and each tool call becomes a visible step and a trace span (Section 9). `MAX_MODEL_STEPS` is the limit that stops endless tool calling.

**Agent instructions** are the `SYSTEM_PROMPT` in the next cell. They explain the role, the tools, when to use RAG and when to use Excel, how to combine both, how to handle missing information, and how to label the sources of each part of the answer.

In [ ]:
TOOLS = [search_knowledge_base, search_server_details, search_incidents,
         get_service_details, get_infrastructure_summary]
TOOLS_BY_NAME = {tool_item.name: tool_item for tool_item in TOOLS}

MAX_MODEL_STEPS = 6                       # maximum LLM turns for one question (prevents endless loops)
llm_with_tools = llm.bind_tools(TOOLS)    # tells the LLM which tools exist and what inputs they take

SYSTEM_PROMPT = '''You are the IT Infrastructure Operations Assistant.
You help IT operations staff with servers, incidents, services and troubleshooting procedures.

Your tools:
- search_knowledge_base: searches the IT operations documents by meaning. Use it for procedures, troubleshooting steps, escalation rules and guidance.
- search_server_details: looks up one server in the Excel Infrastructure sheet (environment, status, utilization, owner).
- search_incidents: finds incidents in the Excel Incidents sheet by server, incident ID, severity or status.
- get_service_details: finds business services in the Excel Service_Inventory sheet (status, criticality, support team).
- get_infrastructure_summary: counts servers by status and summarizes utilization, optionally filtered by environment or status.

How to choose tools:
1. Facts about specific servers, incidents or services come from the Excel tools.
2. Procedures and how-to guidance come from search_knowledge_base.
3. When a question needs both, call both kinds of tool and combine the results.
4. Call the next tool only when you need the result of the previous one. Several independent tool calls may be made in one step.

Rules:
- Never invent server records, incident details, service statuses or procedures.
- If a tool finds nothing, say so clearly and do not guess.
- Keep the answer short and structured. Use these labelled parts when they apply:
  Facts from Excel: (name the tool you used)
  Guidance from knowledge base: (name the document title)
  Recommendation (my reasoning): (your suggestions, based only on the facts above)
'''

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

def run_agent(question):
    '''Run the LLM-and-tools loop for one question. Returns the answer, the tools used and every step.'''
    messages = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=question)]
    tools_used = []
    steps = []

    with tracer.start_as_current_span('Agent Execution') as agent_span:
        agent_span.set_attribute('llm.model', OPENAI_MODEL)
        agent_span.set_attribute('agent.max_model_steps', MAX_MODEL_STEPS)

        for step in range(1, MAX_MODEL_STEPS + 1):
            with tracer.start_as_current_span('LLM Tool Decision') as llm_span:
                llm_span.set_attribute('llm.model', OPENAI_MODEL)
                llm_span.set_attribute('llm.step', step)
                ai_message = llm_with_tools.invoke(messages)
                messages.append(ai_message)
                requested = ai_message.tool_calls or []
                llm_span.set_attribute('llm.tool_calls_requested', len(requested))

                if not requested:
                    # No tool calls means the model has written its final answer.
                    llm_span.update_name('LLM Response Generation')
                    agent_span.set_attribute('agent.steps_used', step)
                    agent_span.set_attribute('agent.tools_used', ', '.join(tools_used))
                    return {'answer': ai_message.content, 'tools_used': tools_used,
                            'steps': steps, 'turns': step}

            # Run every tool the model asked for, then send each result back to the model.
            for call in requested:
                name = call['name']
                args = call['args']
                tools_used.append(name)
                tool_item = TOOLS_BY_NAME.get(name)
                if tool_item is None:
                    result = 'Error: unknown tool ' + name
                else:
                    try:
                        result = tool_item.invoke(args)
                    except Exception as exc:  # the error text goes back to the model
                        result = 'Tool error: ' + str(exc)

                steps.append({'step': step, 'tool': name, 'args': args, 'result': str(result)})
                messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))

        # The loop ended without a final answer, so stop safely.
        agent_span.set_attribute('agent.steps_used', MAX_MODEL_STEPS)
        agent_span.set_attribute('agent.limit_reached', True)
        agent_span.set_attribute('agent.tools_used', ', '.join(tools_used))

    return {'answer': 'Stopped: the step limit was reached before a final answer. Ask a narrower question.',
            'tools_used': tools_used, 'steps': steps, 'turns': MAX_MODEL_STEPS}

In [ ]:
def answer_question(question):
    '''Entry point for one user question. Creates the application's root span.'''
    with tracer.start_as_current_span('IT Infrastructure Operations Assistant') as root:
        root.set_attribute('app.service', SERVICE_NAME)
        with tracer.start_as_current_span('User Query') as query_span:
            cleaned = clean_text(question, limit=300)
            query_span.set_attribute('query.length', len(cleaned))
            query_span.set_attribute('query.preview', cleaned[:60])  # short preview only
            if not cleaned:
                raise ValueError('The question is empty. Type a question and run the cell again.')
        result = run_agent(cleaned)
        root.set_attribute('app.status', 'success')
    return result


def show_run(question, result, preview_chars=250):
    '''Print the question, the tool calls in order, the shortened tool results and the final answer.'''
    print('QUESTION:', question)
    if result['steps']:
        print('\nTool calls, in order:')
        for item in result['steps']:
            print('  step', item['step'], '->', item['tool'], item['args'])
        print('\nTool results (shortened):')
        for item in result['steps']:
            print('  [' + item['tool'] + ']', item['result'][:preview_chars].replace('\n', ' | '))
    else:
        print('\nTool calls: none (the answer came from the model alone)')
    print('\nFINAL ANSWER:')
    print(result['answer'])

In [ ]:
test_question = 'What is the current status of server SRV-104?'
test_result = answer_question(test_question)
show_run(test_question, test_result)

**Expected output:** the agent calls `search_server_details` once, with `server='SRV-104'`, as step 1. The final answer reports status Running, Public Website, CPU 22 percent, memory 40 percent and disk 38 percent, labelled as Facts from Excel. The wording can vary, but the tool path should be the same.

## Section 8: Execute Agentic RAG Scenarios

**Description.** Five scenarios cover knowledge-only, Excel-only and combined questions. For each one, the notebook shows the question, the tools the agent chose in order, the results those tools returned, and the final answer.

| # | Scenario | Expected tool path (the model's choices can differ slightly) |
|---|---|---|
| 1 | Knowledge-only question | search_knowledge_base |
| 2 | Excel-only question | search_server_details |
| 3 | RAG plus Excel question | search_server_details, then search_knowledge_base |
| 4 | Multi-step incident investigation | search_incidents, then search_knowledge_base |
| 5 | Infrastructure summary and recommendations | get_infrastructure_summary, then search_knowledge_base (possibly search_server_details too) |

In [ ]:
SCENARIOS = [
    ('1. Knowledge-only question',
     'What are the recommended steps to troubleshoot a server experiencing high memory usage?'),
    ('2. Excel-only question',
     'Show me the details of server SRV-102, including its current status and environment.'),
    ('3. RAG plus Excel question',
     'Server SRV-102 has high memory usage. Check its infrastructure details, identify the applicable troubleshooting procedure, and recommend the next steps.'),
    ('4. Multi-step incident investigation',
     'Find all critical incidents for server SRV-105 and explain the recommended escalation process.'),
    ('5. Infrastructure summary and recommendations',
     'Identify the infrastructure servers with critical status and explain the first troubleshooting steps.'),
]

scenario_outputs = []
for name, question in SCENARIOS:
    print('=' * 80)
    print(name)
    result = answer_question(question)
    scenario_outputs.append((name, question, result))
    show_run(question, result)
    print()

In [ ]:
summary = pd.DataFrame([
    {
        'Scenario': name,
        'Tools called (in order)': ' -> '.join(result['tools_used']) or 'none',
        'Model turns': result['turns'],
    }
    for name, question, result in scenario_outputs
])
display(summary)

**What to check in the output:**
- Each scenario's tool path should match the table above.
- Scenario 3 uses both sources. Its answer separates Facts from Excel, Guidance from knowledge base and Recommendation (my reasoning).
- Scenario 4 should mention INC-2003 and INC-2004.
- Scenario 5 should name SRV-105, SRV-109 and SRV-114 as Critical.
- Exact wording varies between runs. The facts should not change.

## Section 9: Integrate OpenTelemetry

**Description.** OpenTelemetry records what the application did as **traces** and **spans**.
- **Trace:** one complete request, from the first step to the last. It has one trace ID.
- **Span:** one timed operation inside a trace, for example `Excel Server Lookup`. Each span has a start time, a duration, a status and attributes (details).
- **Parent and child:** a span that starts while another span is open becomes its child. Jaeger uses this to draw the tree.

**What this lab already instruments.** The spans are already written into the code: `IT Infrastructure Operations Assistant`, `User Query`, `Agent Execution`, `LLM Tool Decision`, the Excel and RAG tool spans, and `Response Evaluation` in Section 11. They record nothing until the next cell configures the tracer provider and the exporter. Before that, they are no-ops.

**Export.** The exporter uses **OTLP over HTTP** (the standard OpenTelemetry protocol) and sends spans to `http://localhost:4318`. Jaeger listens on that port when Docker is running. Spans are sent in batches in the background, so the notebook does not wait for each one.

In [ ]:
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

if isinstance(trace.get_tracer_provider(), TracerProvider):
    print('Tracing is already configured in this kernel. Restart the kernel to reset it.')
else:
    resource = Resource.create({'service.name': SERVICE_NAME})   # the name shown in Jaeger
    provider = TracerProvider(resource=resource)
    exporter = OTLPSpanExporter(endpoint=OTLP_ENDPOINT.rstrip('/') + '/v1/traces')
    provider.add_span_processor(BatchSpanProcessor(exporter))    # sends finished spans in groups
    trace.set_tracer_provider(provider)
    print('Tracing configured. Service name:', SERVICE_NAME)
    print('Spans are sent to:', OTLP_ENDPOINT.rstrip('/') + '/v1/traces')
    print('If Jaeger is not running, export messages appear, but the notebook keeps working.')

## Section 10: Explore Traces in Jaeger

**Description.** Jaeger is a trace viewer that runs in Docker. It receives the spans from this notebook and draws them as a timeline.

**Steps:**
1. Open a terminal in the project folder (the folder that contains `docker-compose.yml`).
2. Run `docker compose up -d`. Docker Desktop must be running first.
3. Run the next cell. It checks that the Jaeger web page answers at http://localhost:16686.
4. Run the cell after that. It asks the agent a multi-step question, so a fresh trace is exported.
5. In Jaeger, open the **Search** page. Choose the service `it-infrastructure-agent` and select **Find Traces**. Open the trace named **IT Infrastructure Operations Assistant**.

In [ ]:
import urllib.request

try:
    with urllib.request.urlopen('http://localhost:16686', timeout=3) as response:
        print('Jaeger UI is reachable. HTTP status:', response.status)
except Exception as exc:
    print('Jaeger UI is not reachable yet:', type(exc).__name__)
    print('Start it from the project folder with: docker compose up -d')

In [ ]:
trace_question = scenario_outputs[2][1]   # Scenario 3: RAG plus Excel
print('Running Scenario 3 again so that a fresh trace is exported to Jaeger.')
trace_result = answer_question(trace_question)
print('Tools used:', ' -> '.join(trace_result['tools_used']))

tracer_provider = trace.get_tracer_provider()
if hasattr(tracer_provider, 'force_flush'):
    tracer_provider.force_flush(timeout_millis=10000)   # send the waiting spans now
    print('Spans sent. Open Jaeger and search for service', SERVICE_NAME)
else:
    print('Tracing is not configured yet. Run the Section 9 cell first.')

### Reading the trace, and what is recorded

**Expected trace shape.** The exact steps depend on the model's choices, so yours may differ:

```
IT Infrastructure Operations Assistant      whole question
|-- User Query                               question received and checked
`-- Agent Execution                          the LLM and tools loop
    |-- LLM Tool Decision       step 1        model asks for a tool
    |-- Excel Server Lookup                   tool runs, reports matches
    |-- LLM Tool Decision       step 2        model asks for another tool
    |-- RAG Knowledge Retrieval               vector search, documents retrieved
    `-- LLM Response Generation               final answer, no more tool calls
```

`LLM Response Generation` sits inside `Agent Execution`, because the final answer is written by the last model turn of the loop. The planned diagram showed it outside the agent. `Response Evaluation` appears in the Section 11 traces.

**How to read a span.** Click a span to see its duration, its tags (the attributes, such as `tool.matches`, `rag.documents_retrieved` or `llm.model`) and its status. A span that failed shows an error status and the error message.

**What is recorded:** durations, tool names, number of matching records, number of documents retrieved, model name, step numbers, number of tool calls requested, query length, a 60-character preview of the question, success or error status, and evaluation scores.

**What is not recorded:** API keys, full prompts, full model answers, or any real company data. All lab data is fictional.

## Section 11: Evaluate Agent Responses

**Description.** Evaluation answers the question: is the answer any good? Each answer is scored from 1 (very poor) to 5 (excellent) on four dimensions.

| Dimension | Question it answers | How it is scored |
|---|---|---|
| Relevance | Does the answer address what was asked? | LLM judge (automated) |
| Groundedness | Is each fact supported by the tool results? | LLM judge (automated) |
| Completeness | Does it include the important reference points? | LLM judge (automated) |
| Tool Usage | Did the agent use the right sources and tools? | Code rule (automated, no LLM) |

**Overall** is the average of the four scores.

**Important:** automated scores are not the same as human-verified correctness. The judge can make mistakes, so a person must still check a sample of answers. The table has a *Human verified* column that stays at 'not yet checked' until a person reviews the answer.

In [ ]:
EVAL_SET = [
    {
        'id': 'Q1', 'type': 'Knowledge only',
        'question': 'What are the recommended steps to troubleshoot a server experiencing high memory usage?',
        'expected_tools': ['search_knowledge_base'],
        'reference': 'Use the High Memory Utilization procedure: find the top memory processes, check for a leak, capture diagnostics before a planned restart, and escalate to the application team if memory climbs again.',
    },
    {
        'id': 'Q2', 'type': 'Excel only',
        'question': 'Show me the details of server SRV-102, including its current status and environment.',
        'expected_tools': ['search_server_details'],
        'reference': 'SRV-102 (APP-ORDERS-02) is in the Production environment, has status Warning and memory utilization of 93 percent.',
    },
    {
        'id': 'Q3', 'type': 'RAG plus Excel',
        'question': 'Server SRV-102 has high memory usage. Check its infrastructure details, identify the applicable troubleshooting procedure, and recommend the next steps.',
        'expected_tools': ['search_server_details', 'search_knowledge_base'],
        'reference': 'Facts: SRV-102 is Production, status Warning, memory 93 percent. Guidance: High Memory Utilization Troubleshooting. Recommendations must be labelled as the assistant reasoning.',
    },
    {
        'id': 'Q4', 'type': 'Multi-step incidents',
        'question': 'Find all critical incidents for server SRV-105 and explain the recommended escalation process.',
        'expected_tools': ['search_incidents', 'search_knowledge_base'],
        'reference': 'Critical incidents for SRV-105 are INC-2003 (payment authorization failures, Open) and INC-2004 (CPU saturation during batch settlement, In Progress). Escalation comes from the Incident Escalation Procedure, which says Critical incidents go to the on-call manager at once.',
    },
    {
        'id': 'Q5', 'type': 'Summary plus guidance',
        'question': 'Identify the infrastructure servers with critical status and explain the first troubleshooting steps.',
        'expected_tools': ['get_infrastructure_summary', 'search_knowledge_base'],
        'reference': 'The Critical servers are SRV-105, SRV-109 and SRV-114. First steps come from the matching troubleshooting documents, such as High CPU Utilization and Disk Space Management.',
    },
    {
        'id': 'Q6', 'type': 'Missing information',
        'question': 'What is the status of server SRV-999, and what is the backup procedure for it?',
        'expected_tools': ['search_server_details', 'search_knowledge_base'],
        'reference': 'SRV-999 does not exist in the workbook. The answer must say so and must not invent a status. A general backup procedure can come from the knowledge base, but no server-specific facts.',
    },
]

print('Evaluation questions:', len(EVAL_SET))

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from opentelemetry.trace import Status, StatusCode

JUDGE_INSTRUCTIONS = '''You grade the answer of an IT operations assistant. Score three dimensions from 1 (very poor) to 5 (excellent):
- relevance: does the answer address the user question?
- groundedness: are the facts supported by the tool evidence? Heavily penalize any server, incident, service or procedure that the evidence does not contain.
- completeness: does the answer include the important points of the reference?
Reply with one JSON object and no other text. It must contain the integer keys relevance, groundedness and completeness, plus the text keys relevance_reason, groundedness_reason and completeness_reason (one short sentence each).
'''


def judge_answer(item, result):
    '''Ask the LLM judge for three scores. Returns the scores, or an error note if the judge reply cannot be read.'''
    evidence = '\n\n'.join('[' + s['tool'] + '] ' + s['result'] for s in result['steps'])
    if not evidence:
        evidence = '(no tools were called)'
    prompt = ('Question: ' + item['question'] + '\n'
              'Reference: ' + item['reference'] + '\n\n'
              'Tool evidence:\n' + evidence[:3000] + '\n\n'
              'Assistant answer:\n' + str(result['answer']))

    with tracer.start_as_current_span('Response Evaluation') as span:
        span.set_attribute('eval.question_id', item['id'])
        span.set_attribute('eval.judge_model', OPENAI_MODEL)
        reply = llm.invoke([SystemMessage(content=JUDGE_INSTRUCTIONS), HumanMessage(content=prompt)])
        raw = str(reply.content)
        try:
            scores = json.loads(raw[raw.find('{'):raw.rfind('}') + 1])
        except ValueError as exc:
            span.set_attribute('eval.parse_error', True)
            span.set_status(Status(StatusCode.ERROR, 'judge reply could not be read'))
            return {'judge_error': 'Judge reply could not be read: ' + str(exc)}
        span.set_attribute('eval.parse_error', False)
        return scores


def score_tool_usage(expected, used):
    '''Code rule, not an LLM. Reward using the expected tools and lightly penalize extra tools. Returns 1 to 5.'''
    expected_set = set(expected)
    used_set = set(used)
    coverage = len(expected_set & used_set) / len(expected_set) if expected_set else 1.0
    extra = len(used_set - expected_set)
    raw_score = 1 + 4 * coverage - min(extra, 2)
    return max(1, min(5, round(raw_score)))

In [ ]:
def answer_and_evaluate(item):
    '''Run one evaluation question inside a single trace: answer, judge, and score the tool usage.'''
    with tracer.start_as_current_span('IT Infrastructure Operations Assistant') as root:
        root.set_attribute('eval.question_id', item['id'])
        with tracer.start_as_current_span('User Query') as query_span:
            query_span.set_attribute('query.length', len(item['question']))
            query_span.set_attribute('query.preview', item['question'][:60])
        result = run_agent(item['question'])
        judge = judge_answer(item, result)
        tool_score = score_tool_usage(item['expected_tools'], result['tools_used'])
    return result, judge, tool_score


def to_score(value):
    '''Convert a judge value to a whole-number score, or None if it is not a number.'''
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


score_rows = []
reason_rows = []
for item in EVAL_SET:
    print('Evaluating', item['id'], '...')
    result, judge, tool_score = answer_and_evaluate(item)

    relevance = to_score(judge.get('relevance'))
    groundedness = to_score(judge.get('groundedness'))
    completeness = to_score(judge.get('completeness'))
    four_scores = [relevance, groundedness, completeness, tool_score]
    overall = round(sum(four_scores) / 4, 2) if None not in four_scores else None

    score_rows.append({
        'Question': item['id'] + ' ' + item['question'][:50] + '...',
        'Relevance': relevance,
        'Groundedness': groundedness,
        'Completeness': completeness,
        'Tool Usage': tool_score,
        'Overall': overall,
        'Tools used': ', '.join(result['tools_used']) or 'none',
        'Human verified': 'not yet checked',
    })
    reason_rows.append({
        'Question': item['id'],
        'Relevance reason': judge.get('relevance_reason', judge.get('judge_error', '')),
        'Groundedness reason': judge.get('groundedness_reason', ''),
        'Completeness reason': judge.get('completeness_reason', ''),
    })

print()
print('Scores (1 = very poor, 5 = excellent). Relevance, Groundedness and Completeness come from the LLM judge.')
display(pd.DataFrame(score_rows))
print('\nJudge reasons:')
display(pd.DataFrame(reason_rows))

**How to use these results:**
1. Find any low score (1 or 2). Read the judge reason and the tool path. A Tool Usage score below 4 means the agent skipped a source or used an unexpected one.
2. Check at least two answers yourself against the workbook and the knowledge base. Record your verdict in the *Human verified* column of your own copy of the results.
3. Improve the assistant where the evidence points: clearer tool descriptions, a better system prompt, or a missing knowledge-base document.

**Why evaluation matters for enterprise GenAI.** A fluent answer can still be wrong. An assistant that invents a server status or a procedure can send an engineer to the wrong system during an outage. Repeatable evaluation with a fixed question set shows whether a change made the assistant better or worse, before users see it. Pair automated scores with human review, and keep the evaluation set up to date as the infrastructure changes.

## Section 12: Summary and Knowledge Check

**Architecture recap**
- **Knowledge base (RAG):** 12 IT procedures are embedded with OpenAI embeddings and stored in a local FAISS index. The `search_knowledge_base` tool returns the closest documents by meaning.
- **Excel tools:** four read-only Pandas tools answer structured questions about servers, incidents, services and utilization summaries.
- **Agent reasoning:** the LLM reads the question and the tool descriptions, chooses tools, reads the results, and repeats until it can answer. The loop has a step limit.
- **Observability:** OpenTelemetry spans show each step, its duration, its tool and its status. Jaeger displays them as a tree.
- **Evaluation:** a fixed question set, a 1 to 5 rubric, an LLM judge for three dimensions, a code rule for tool usage, and human review for final assurance.

**Limits of this lab:** the data is fictional, the tools are read-only, the knowledge base is small, and the judge is an LLM, so its scores need human checks.

### Knowledge check (5 questions)

**1. What is the main difference between traditional RAG and agentic RAG?**
- a) Agentic RAG does not use documents.
- b) Agentic RAG lets the LLM choose which sources and tools to call, and in what order.
- c) Agentic RAG stores all data in Excel.
- d) Agentic RAG removes the need for an LLM.

**2. What does a FAISS index store, and what does a lower distance mean?**
- a) Passwords. A lower distance means a more secure match.
- b) Vectors (embeddings) of the documents. A lower distance means the document is closer in meaning to the question.
- c) Excel rows. A lower distance means an older row.
- d) Trace spans. A lower distance means a faster response.

**3. Why does the tool docstring matter?**
- a) It is only documentation for developers and the LLM never sees it.
- b) The LLM reads the name, the description and the input types to decide when and how to call the tool.
- c) It sets the memory limit for the tool.
- d) It is the password for the Excel file.

**4. In Jaeger, what is a span?**
- a) A whole user session.
- b) One timed operation inside a trace, such as an Excel lookup, with attributes and a status.
- c) The Docker container that runs Jaeger.
- d) An OpenAI API key.

**5. Which statement about evaluation scores is correct?**
- a) Automated judge scores replace human checking.
- b) Automated judge scores are useful for spotting trends, but important answers still need human verification.
- c) A score of 5 is guaranteed whenever the agent calls a tool.
- d) Evaluation is only needed after the application is deployed.

---

**Answers and explanations**
1. **b.** Agentic RAG adds decision-making. The agent chooses sources and tools and can repeat steps. Traditional RAG follows one fixed search-then-answer path.
2. **b.** FAISS stores the embedding vectors of the documents. Its distance measures how far a document vector is from the question vector, so a lower value means a closer match.
3. **b.** The docstring, together with the function name and typed inputs, is the tool description the LLM uses. A vague description leads to wrong or missing tool calls.
4. **b.** A span is one timed step with a name, duration, attributes and status. Spans with a parent-child relationship form the tree that Jaeger displays.
5. **b.** Automated scores are fast and repeatable, which makes them good for comparing versions. They can still be wrong, so people must check important answers.